# Synapse → Databricks Table Migration

Reads table definitions from Synapse Analytics, creates matching tables in Databricks, and ingests data.  
Runs locally using `pyodbc` (ActiveDirectoryInteractive) and `databricks-sql-connector`.

In [1]:
import pyodbc
from databricks import sql
import polars as pl
import math
from datetime import datetime, date, time
from decimal import Decimal

# ─── Synapse Connection ────────────────────────────────────────────────────────
synapse_server = "syn-rt-prd-unity.sql.azuresynapse.net"
synapse_database = "syndwrtprdunity"
synapse_username = "byambadorjme@riotinto.com"
synapse_driver = "{ODBC Driver 17 for SQL Server}"
synapse_source_schema = "MDE"

synapse_connection_string = f"""
    DRIVER={synapse_driver};
    SERVER={synapse_server};
    DATABASE={synapse_database};
    UID={synapse_username};
    Authentication=ActiveDirectoryInteractive;
    Encrypt=yes;
    TrustServerCertificate=no;
    Connection Timeout=30;
"""

# ─── Databricks Connection ─────────────────────────────────────────────────────
databricks_access_token = "os.environ["DATABRICKS_TOKEN"]"
databricks_server_hostname = "adb-2439498039309203.3.azuredatabricks.net"
databricks_http_path = "/sql/1.0/warehouses/e7d134c4348ac8b5"
databricks_target_catalog = "_ogr_azr_syd_dev"       # TODO: change to your target catalog
databricks_target_schema = "mde"                  # TODO: change to your target schema

# ─── Tables to migrate (schema.table from Synapse) ─────────────────────────────
tables_to_migrate = [
    "MDE.midas_annualplan",
    "MDE.midas_basemetric",
    "MDE.midas_dataflow",
    "MDE.midas_dataflowdependency",
    "MDE.midas_eedrillingprojectlocations",
    "MDE.midas_eedrillingprojectlocationspattern",
    "MDE.midas_expitmovementactual",
    "MDE.midas_fixedplantmetricmapping",
    "MDE.midas_hmeassetactivitytype",
    "MDE.midas_hmeassetmapping",
    "MDE.midas_hmetummapping",
    "MDE.midas_manualbasemetrics",
    "MDE.midas_mdr",
    "MDE.midas_primegreatersites",
    "MDE.midas_publicholiday",
    "MDE.midas_recovery",
    "MDE.midas_sitemapping",
    "MDE.midas_sourcetable",
    "MDE.midas_spatialconformance",
    "MDE.gmr_basemeasure",
    "MDE.gmr_basemeasuremapping",
    "MDE.gmr_metric",
    "MDE.gmr_metricmapping",
    "MDE.gmr_metriccategory",
    "MDE.gmr_environment"
]

# ─── Batch size for inserts ─────────────────────────────────────────────────────
BATCH_SIZE = 500

In [2]:
# ─── Databricks Environments ───────────────────────────────────────────────────
environments = [
    {
        "name": "dev",
        "access_token": "os.environ["DATABRICKS_DEV_TOKEN"]",
        "server_hostname": "adb-7405619384118055.15.azuredatabricks.net",
        "http_path": "/sql/1.0/warehouses/83b94ea73376738f",
        "catalog": "_ogr_azr_syd_dev",
        "schema": "mde",
    },
    {
        "name": "tst",
        "access_token": "os.environ["DATABRICKS_TST_TOKEN"]",
        "server_hostname": "adb-7405606707500852.12.azuredatabricks.net",
        "http_path": "/sql/1.0/warehouses/4e68d63969c13d07",
        "catalog": "_ogr_azr_syd_tst",
        "schema": "mde",
    },
    {
        "name": "prd",
        "access_token": "os.environ["DATABRICKS_PRD_TOKEN"]",
        "server_hostname": "adb-7405610406720068.8.azuredatabricks.net",
        "http_path": "/sql/1.0/warehouses/31f7b82452828725",
        "catalog": "_ogr_azr_syd",
        "schema": "mde",
    },
]

In [3]:
# ─── Establish Synapse Connection ───────────────────────────────────────────────
print("Connecting to Synapse (browser auth prompt may appear)...")
synapse_conn = pyodbc.connect(synapse_connection_string)
print("✓ Synapse connected")

Connecting to Synapse (browser auth prompt may appear)...
✓ Synapse connected


In [4]:
# ─── Type Mapping & Helper Functions ───────────────────────────────────────────

SYNAPSE_TO_DATABRICKS_TYPE = {
    "bigint": "BIGINT",
    "int": "INT",
    "smallint": "SMALLINT",
    "tinyint": "TINYINT",
    "bit": "BOOLEAN",
    "float": "DOUBLE",
    "real": "FLOAT",
    "decimal": "DECIMAL",
    "numeric": "DECIMAL",
    "money": "DECIMAL(19,4)",
    "smallmoney": "DECIMAL(10,4)",
    "varchar": "STRING",
    "nvarchar": "STRING",
    "char": "STRING",
    "nchar": "STRING",
    "text": "STRING",
    "ntext": "STRING",
    "date": "DATE",
    "datetime": "TIMESTAMP",
    "datetime2": "TIMESTAMP",
    "smalldatetime": "TIMESTAMP",
    "datetimeoffset": "STRING",
    "time": "STRING",
    "uniqueidentifier": "STRING",
    "varbinary": "BINARY",
    "binary": "BINARY",
    "image": "BINARY",
    "xml": "STRING",
    "sql_variant": "STRING",
}


def get_databricks_type(synapse_type, num_precision, num_scale):
    """Map a Synapse data type to a Databricks SQL type."""
    base = synapse_type.strip().lower()
    if base in ("decimal", "numeric"):
        p = num_precision if num_precision is not None else 38
        s = num_scale if num_scale is not None else 0
        return f"DECIMAL({p},{s})"
    return SYNAPSE_TO_DATABRICKS_TYPE.get(base, "STRING")


def format_sql_value(val):
    """Format a Python value as a Databricks SQL literal."""
    if val is None:
        return "NULL"
    if isinstance(val, bool):
        return "TRUE" if val else "FALSE"
    if isinstance(val, int):
        return str(val)
    if isinstance(val, float):
        if math.isnan(val) or math.isinf(val):
            return "NULL"
        return repr(val)
    if isinstance(val, Decimal):
        return str(val)
    if isinstance(val, datetime):
        return f"'{val.strftime('%Y-%m-%d %H:%M:%S.%f')}'"
    if isinstance(val, date):
        return f"'{val.isoformat()}'"
    if isinstance(val, time):
        return f"'{val.isoformat()}'"
    if isinstance(val, (bytes, bytearray)):
        return f"X'{val.hex()}'"
    # Default: treat as string, escape single quotes
    escaped = str(val).replace("'", "''")
    return f"'{escaped}'"

In [5]:
def migrate_table(
    synapse_conn,
    databricks_cursor,
    synapse_schema: str,
    synapse_table: str,
    dbx_catalog: str,
    dbx_schema: str,
    dbx_table: str,
    batch_size: int = 500,
):
    """Read schema + data from Synapse, create table in Databricks, and insert rows."""

    # ── 1. Read column metadata from Synapse ──
    schema_query = f"""
        SELECT COLUMN_NAME, DATA_TYPE,
               NUMERIC_PRECISION, NUMERIC_SCALE,
               ORDINAL_POSITION
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = '{synapse_schema}'
          AND TABLE_NAME   = '{synapse_table}'
        ORDER BY ORDINAL_POSITION
    """
    schema_df = pl.read_database(schema_query, synapse_conn)

    if schema_df.height == 0:
        print(f"  ⚠ No columns found for {synapse_schema}.{synapse_table} — skipping")
        return {"table": synapse_table, "status": "skipped", "reason": "no columns"}

    col_names = schema_df["COLUMN_NAME"].to_list()

    # ── 2. Build and execute CREATE TABLE DDL ──
    col_defs = []
    for row in schema_df.iter_rows(named=True):
        dbx_type = get_databricks_type(
            row["DATA_TYPE"], row["NUMERIC_PRECISION"], row["NUMERIC_SCALE"]
        )
        col_defs.append(f"    `{row['COLUMN_NAME']}` {dbx_type}")

    full_table = f"`{dbx_catalog}`.`{dbx_schema}`.`{dbx_table}`"
    ddl = f"CREATE OR REPLACE TABLE {full_table} (\n" + ",\n".join(col_defs) + "\n)"

    print(f"  Creating table {full_table} ({len(col_names)} columns)...")
    databricks_cursor.execute(ddl)

    # ── 3. Read all data from Synapse ──
    cols_sql = ", ".join(f"[{c}]" for c in col_names)
    data_query = f"SELECT {cols_sql} FROM [{synapse_schema}].[{synapse_table}]"

    print(f"  Reading data from {synapse_schema}.{synapse_table}...")
    data_df = pl.read_database(data_query, synapse_conn)
    total_rows = data_df.height
    print(f"  Fetched {total_rows:,} rows")

    if total_rows == 0:
        print(f"  ✓ Table created (empty)")
        return {"table": synapse_table, "status": "ok", "rows": 0}

    # ── 4. Insert in batches ──
    rows = data_df.rows()
    inserted = 0

    for i in range(0, total_rows, batch_size):
        batch = rows[i : i + batch_size]
        values_clauses = []
        for row in batch:
            formatted = ", ".join(format_sql_value(v) for v in row)
            values_clauses.append(f"({formatted})")

        insert_sql = f"INSERT INTO {full_table} VALUES\n" + ",\n".join(values_clauses)
        databricks_cursor.execute(insert_sql)
        inserted += len(batch)
        print(f"  Inserted {inserted:,}/{total_rows:,} rows", end="\r")

    print(f"  ✓ Inserted {inserted:,}/{total_rows:,} rows      ")
    return {"table": synapse_table, "status": "ok", "rows": inserted}

In [6]:
# ─── Run Migration for All Tables across All Environments ──────────────────────
all_results = {}

for env in environments:
    env_name = env["name"]
    print(f"\n{'═' * 60}")
    print(f"  ENVIRONMENT: {env_name.upper()} ({env['catalog']})")
    print(f"{'═' * 60}")

    # Connect to this environment's Databricks
    print(f"  Connecting to Databricks {env_name}...")
    dbx_conn = sql.connect(
        server_hostname=env["server_hostname"],
        http_path=env["http_path"],
        access_token=env["access_token"],
    )
    dbx_cursor = dbx_conn.cursor()
    print(f"  ✓ Connected")

    # Ensure target schema exists
    dbx_cursor.execute(
        f"CREATE SCHEMA IF NOT EXISTS `{env['catalog']}`.`{env['schema']}`"
    )
    print(f"  ✓ Schema {env['catalog']}.{env['schema']} ready")

    # Migrate each table
    results = []
    for i, qualified_name in enumerate(tables_to_migrate, 1):
        syn_schema, syn_table = qualified_name.split(".", 1)
        dbx_table = syn_table

        print(f"\n  [{i}/{len(tables_to_migrate)}] {qualified_name}")
        try:
            result = migrate_table(
                synapse_conn=synapse_conn,
                databricks_cursor=dbx_cursor,
                synapse_schema=syn_schema,
                synapse_table=syn_table,
                dbx_catalog=env["catalog"],
                dbx_schema=env["schema"],
                dbx_table=dbx_table,
                batch_size=BATCH_SIZE,
            )
            results.append(result)
        except Exception as e:
            print(f"    ✗ ERROR: {e}")
            results.append({"table": syn_table, "status": "error", "reason": str(e)})

    # Close this environment's connection
    dbx_cursor.close()
    dbx_conn.close()
    all_results[env_name] = results

# ─── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'═' * 60}")
print("Migration Summary")
print(f"{'═' * 60}")
for env_name, results in all_results.items():
    ok = sum(1 for r in results if r.get("status") == "ok")
    err = sum(1 for r in results if r.get("status") == "error")
    skip = sum(1 for r in results if r.get("status") == "skipped")
    print(f"  {env_name.upper():4s}: {ok} ok, {err} errors, {skip} skipped")
print()
summary_df = pl.DataFrame(all_results["dev"])  # show dev detail
print(summary_df)


════════════════════════════════════════════════════════════
  ENVIRONMENT: DEV (_ogr_azr_syd_dev)
════════════════════════════════════════════════════════════
  Connecting to Databricks dev...
  ✓ Connected
  ✓ Schema _ogr_azr_syd_dev.mde ready

  [1/25] MDE.midas_annualplan
  Creating table `_ogr_azr_syd_dev`.`mde`.`midas_annualplan` (12 columns)...
  Reading data from MDE.midas_annualplan...
  Fetched 7,836 rows
  ✓ Inserted 7,836/7,836 rows      

  [2/25] MDE.midas_basemetric
  Creating table `_ogr_azr_syd_dev`.`mde`.`midas_basemetric` (52 columns)...
  Reading data from MDE.midas_basemetric...
  Fetched 398 rows
  ✓ Inserted 398/398 rows      

  [3/25] MDE.midas_dataflow
  Creating table `_ogr_azr_syd_dev`.`mde`.`midas_dataflow` (15 columns)...
  Reading data from MDE.midas_dataflow...
  Fetched 590 rows
  ✓ Inserted 590/590 rows      

  [4/25] MDE.midas_dataflowdependency
  Creating table `_ogr_azr_syd_dev`.`mde`.`midas_dataflowdependency` (9 columns)...
  Reading data from M

## Generate Power Automate Sync Configuration Table
Queries Synapse `INFORMATION_SCHEMA.COLUMNS` for all 25 MDE tables and builds the config rows:
`sync_table_name`, `last_sync_timestamp`, `databricks_target_table`, `column_list`, `primary_key_column`, `phantom_row_sql`

In [7]:
# ─── Query all column metadata from Synapse in one shot ────────────────────────
table_names_sql = ", ".join(
    f"'{t.split('.', 1)[1]}'" for t in tables_to_migrate
)

all_columns_query = f"""
    SELECT
        TABLE_NAME,
        COLUMN_NAME,
        DATA_TYPE,
        NUMERIC_PRECISION,
        NUMERIC_SCALE,
        ORDINAL_POSITION
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = '{synapse_source_schema}'
      AND TABLE_NAME IN ({table_names_sql})
    ORDER BY TABLE_NAME, ORDINAL_POSITION
"""

all_columns_df = pl.read_database(all_columns_query, synapse_conn)
print(f"Fetched {all_columns_df.height} column definitions across "
      f"{all_columns_df['TABLE_NAME'].n_unique()} tables")


# ─── Map Synapse type → Databricks SQL literal type ────────────────────────────
def to_dbx_type(data_type, num_precision, num_scale):
    base = data_type.strip().lower()
    if base in ("decimal", "numeric"):
        p = num_precision if num_precision is not None else 38
        s = num_scale if num_scale is not None else 0
        return f"DECIMAL({p},{s})"
    return SYNAPSE_TO_DATABRICKS_TYPE.get(base, "STRING")


# ─── Build config rows ────────────────────────────────────────────────────────
DBX_CATALOG = "_ogr_azr_syd_dev"
DBX_SCHEMA  = "mde"

config_rows = []

for qualified in tables_to_migrate:
    _, table = qualified.split(".", 1)

    cols_df = (
        all_columns_df
        .filter(pl.col("TABLE_NAME") == table)
        .sort("ORDINAL_POSITION")
    )

    if cols_df.height == 0:
        print(f"⚠  {table}: no columns found — skipped")
        continue

    # Column names (alphabetically sorted, matching the example)
    col_names = sorted(cols_df["COLUMN_NAME"].str.to_lowercase().to_list())

    # Primary key — Dataverse convention: <tablename>id
    pk = f"{table}id"
    if pk not in col_names:
        # fallback: first column whose name ends with 'id'
        id_cols = [c for c in col_names if c.endswith("id")]
        pk = id_cols[0] if id_cols else col_names[0]

    # Sync table name — Dataverse entity-set name is logical name + 's'
    sync_name = table + "s"

    # Databricks target
    dbx_target = f"{DBX_CATALOG}.{DBX_SCHEMA}.{table}"

    # Phantom row SQL
    # Build in column_list (alphabetical) order so positions match
    cast_parts = []
    for c in col_names:
        row = cols_df.filter(pl.col("COLUMN_NAME").str.to_lowercase() == c).row(0, named=True)
        dbx_type = to_dbx_type(row["DATA_TYPE"], row["NUMERIC_PRECISION"], row["NUMERIC_SCALE"])
        cast_parts.append(f"CAST(NULL AS {dbx_type}) AS {c}")

    phantom_sql = "SELECT " + ", ".join(cast_parts) + " WHERE 1=0"

    config_rows.append({
        "sync_table_name": sync_name,
        "last_sync_timestamp": "",
        "databricks_target_table": dbx_target,
        "column_list": ", ".join(col_names),
        "primary_key_column": pk,
        "phantom_row_sql": phantom_sql,
    })

config_df = pl.DataFrame(config_rows)
print(f"\n✓ Generated {len(config_rows)} config entries")
config_df.select("sync_table_name", "primary_key_column")

Fetched 436 column definitions across 25 tables

✓ Generated 25 config entries


sync_table_name,primary_key_column
str,str
"""midas_annualplans""","""midas_annualplanid"""
"""midas_basemetrics""","""midas_basemetricid"""
"""midas_dataflows""","""midas_dataflowid"""
"""midas_dataflowdependencys""","""midas_dataflowdependencyid"""
"""midas_eedrillingprojectlocatio…","""midas_eedrillingprojectlocatio…"
…,…
"""gmr_basemeasuremappings""","""gmr_basemeasuremappingid"""
"""gmr_metrics""","""gmr_metricid"""
"""gmr_metricmappings""","""gmr_metricmappingid"""


In [8]:
# ─── Print each config entry in a readable format ─────────────────────────────
pl.Config.set_tbl_cols(10)
pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_width_chars(300)

for row in config_df.iter_rows(named=True):
    print(row["sync_table_name"])
    print()
    print(row["last_sync_timestamp"] or "(set at runtime)")
    print()
    print(row["databricks_target_table"])
    print()
    print(row["column_list"])
    print()
    print(row["primary_key_column"])
    print()
    print(row["phantom_row_sql"])
    print()
    print("─" * 120)
    print()

midas_annualplans

(set at runtime)

_ogr_azr_syd_dev.mde.midas_annualplan

createdon, midas_annualplan, midas_annualplanid, midas_effectivedatefrom, midas_environment, midas_planmonth, midas_plannamenew, midas_plannedvalue, midas_plansource, midas_sitecode, modifiedon, statecode

midas_annualplanid

SELECT CAST(NULL AS TIMESTAMP) AS createdon, CAST(NULL AS STRING) AS midas_annualplan, CAST(NULL AS STRING) AS midas_annualplanid, CAST(NULL AS TIMESTAMP) AS midas_effectivedatefrom, CAST(NULL AS INT) AS midas_environment, CAST(NULL AS TIMESTAMP) AS midas_planmonth, CAST(NULL AS STRING) AS midas_plannamenew, CAST(NULL AS DECIMAL(38,18)) AS midas_plannedvalue, CAST(NULL AS STRING) AS midas_plansource, CAST(NULL AS STRING) AS midas_sitecode, CAST(NULL AS TIMESTAMP) AS modifiedon, CAST(NULL AS INT) AS statecode WHERE 1=0

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

midas_basemetrics

(set at runtime)

_ogr_azr_syd_d

In [9]:
# ─── Export to Excel (optional) ──────────────────────────────────────────────────
output_path = "sync_config_table.xlsx"
config_df.write_excel(output_path)
print(f"✓ Exported to {output_path}")

✓ Exported to sync_config_table.xlsx


## Rebuild Config from Dataverse API
Connect directly to Dataverse to get the correct **EntitySetName** (sync_table_name) and attribute types for each entity, then cross-reference with the Synapse column list for precision.

In [13]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ─── Authenticate to Dataverse ─────────────────────────────────────────────────
dataverse_url = "https://rt-citizen-prime-prd-au.crm6.dynamics.com"
client_id     = "c249f153-f8f1-44c5-8c3b-5e7368635f8d"
tenant_id     = "4341df80-fbe6-41bf-89b0-e6e2379c9c23"
client_secret = "os.environ["DATAVERSE_CLIENT_SECRET"]"

token_resp = requests.post(
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
    data={
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
        "scope": f"{dataverse_url}/.default",
    },
)
token_resp.raise_for_status()
dv_token = token_resp.json()["access_token"]
print("✓ Authenticated to Dataverse")

dv_session = requests.Session()
dv_session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)))
dv_headers = {
    "Authorization": f"Bearer {dv_token}",
    "Accept": "application/json",
    "OData-Version": "4.0",
    "OData-MaxVersion": "4.0",
}

# ─── Fetch all EntityDefinitions → EntitySetName lookup ────────────────────────
print("Fetching EntityDefinitions (may take a moment)...")
ent_resp = dv_session.get(
    f"{dataverse_url}/api/data/v9.2/EntityDefinitions",
    headers=dv_headers,
    timeout=(10, 120),
)
ent_resp.raise_for_status()
entity_set_map = {
    e["LogicalName"]: e.get("EntitySetName", e["LogicalName"] + "s")
    for e in ent_resp.json()["value"]
}
print(f"  Loaded {len(entity_set_map)} entity definitions")

✓ Authenticated to Dataverse
Fetching EntityDefinitions (may take a moment)...
  Loaded 942 entity definitions


In [ ]:
# ─── Dataverse AttributeType → Databricks SQL type ────────────────────────────
DV_TYPE_MAP = {
    "String": "STRING",
    "Memo": "STRING",
    "Integer": "INT",
    "BigInt": "BIGINT",
    "Decimal": "DECIMAL",       # precision filled from Synapse or default
    "Double": "DOUBLE",
    "Money": "DECIMAL",         # precision filled from Synapse or default
    "Boolean": "BOOLEAN",
    "DateTime": "TIMESTAMP",
    "Uniqueidentifier": "STRING",
    "Lookup": "STRING",
    "Customer": "STRING",
    "Owner": "STRING",
    "Picklist": "INT",
    "State": "INT",
    "Status": "INT",
    "EntityName": "STRING",
    "Image": "STRING",
    "File": "STRING",
    "MultiSelectPicklist": "STRING",
}

SKIP_ATTR_TYPES = {"Virtual", "ManagedProperty", "CalendarRules", "PartyList"}

# ─── Build config from Dataverse + Synapse precision ──────────────────────────
config_rows_dv = []
table_names = [t.split(".", 1)[1] for t in tables_to_migrate]

for table in table_names:
    entity_set_name = entity_set_map.get(table)
    if not entity_set_name:
        print(f"⚠ {table}: not found in Dataverse — skipped")
        continue

    # Fetch attributes for this entity
    attrs_resp = dv_session.get(
        f"{dataverse_url}/api/data/v9.2/"
        f"EntityDefinitions(LogicalName='{table}')/Attributes",
        headers=dv_headers,
        timeout=(10, 90),
    )
    attrs_resp.raise_for_status()
    raw_attrs = attrs_resp.json()["value"]

    # Synapse columns for this table (lowercase set for cross-reference)
    syn_cols_for_table = set(
        all_columns_df
        .filter(pl.col("TABLE_NAME") == table)["COLUMN_NAME"]
        .str.to_lowercase()
        .to_list()
    )

    # Filter & map attributes
    attrs = []
    for a in raw_attrs:
        attr_type = a.get("AttributeType", "")
        name = a.get("LogicalName", "")

        if attr_type in SKIP_ATTR_TYPES:
            continue

        # Skip dependent/companion attributes (e.g., lookup "name" fields)
        # These exist in metadata but are NOT queryable via $select
        if a.get("AttributeOf"):
            continue

        # Only keep columns that exist in the Synapse MDE schema
        if syn_cols_for_table and name.lower() not in syn_cols_for_table:
            continue

        # Lookup/Customer/Owner attributes use _<logicalname>_value in Web API $select
        if attr_type in ("Lookup", "Customer", "Owner"):
            api_name = f"_{name}_value"
        else:
            api_name = name

        # Map to Databricks type
        dbx_type = DV_TYPE_MAP.get(attr_type, "STRING")

        # For Decimal/Money: pull precision from Synapse metadata
        if dbx_type == "DECIMAL":
            syn_row = all_columns_df.filter(
                (pl.col("TABLE_NAME") == table)
                & (pl.col("COLUMN_NAME").str.to_lowercase() == name.lower())
            )
            if syn_row.height > 0:
                p = syn_row["NUMERIC_PRECISION"][0]
                s = syn_row["NUMERIC_SCALE"][0]
                if p is not None:
                    dbx_type = f"DECIMAL({p},{s or 0})"
                else:
                    dbx_type = "DECIMAL(38,4)" if attr_type == "Money" else "DECIMAL(38,10)"
            else:
                dbx_type = "DECIMAL(38,4)" if attr_type == "Money" else "DECIMAL(38,10)"

        attrs.append({"api_name": api_name, "dbx_name": name, "dbx_type": dbx_type})

    if not attrs:
        print(f"⚠ {table}: no matching columns — skipped")
        continue

    # Sort alphabetically by Databricks column name
    attrs.sort(key=lambda x: x["dbx_name"])
    api_names = [a["api_name"] for a in attrs]
    dbx_names = [a["dbx_name"] for a in attrs]

    # Primary key — Dataverse convention: <tablename>id
    pk = f"{table}id"
    if pk not in dbx_names:
        id_cols = [c for c in dbx_names if c.endswith("id")]
        pk = id_cols[0] if id_cols else dbx_names[0]

    # Phantom row SQL — uses Databricks (Synapse) column names
    cast_parts = [f"CAST(NULL AS {a['dbx_type']}) AS {a['dbx_name']}" for a in attrs]
    phantom_sql = "SELECT " + ", ".join(cast_parts) + " WHERE 1=0"

    dbx_target = f"{DBX_CATALOG}.{DBX_SCHEMA}.{table}"

    config_rows_dv.append({
        "sync_table_name": entity_set_name,
        "last_sync_timestamp": "",
        "databricks_target_table": dbx_target,
        "column_list": ", ".join(api_names),
        "primary_key_column": pk,
        "phantom_row_sql": phantom_sql,
        "column_mapping": str(dict(zip(api_names, dbx_names))),
    })


    print(f"✓ {table} → {entity_set_name} ({len(col_names)} cols, "
          f"{len(raw_attrs)} total in Dataverse)")

config_dv_df = pl.DataFrame(config_rows_dv)

print(f"\n✓ Generated {len(config_rows_dv)} config entries from Dataverse")config_dv_df.select("sync_table_name", "primary_key_column")

✓ midas_annualplan → midas_annualplans (12 cols, 41 total in Dataverse)
✓ midas_basemetric → midas_basemetrics (44 cols, 100 total in Dataverse)
✓ midas_dataflow → midas_dataflows (15 cols, 45 total in Dataverse)
✓ midas_dataflowdependency → midas_dataflowdependencies (7 cols, 37 total in Dataverse)
✓ midas_eedrillingprojectlocations → midas_eedrillingprojectlocationses (8 cols, 37 total in Dataverse)
✓ midas_eedrillingprojectlocationspattern → midas_eedrillingprojectlocationspatterns (9 cols, 40 total in Dataverse)
✓ midas_expitmovementactual → midas_expitmovementactuals (10 cols, 39 total in Dataverse)
✓ midas_fixedplantmetricmapping → midas_fixedplantmetricmappings (11 cols, 40 total in Dataverse)
✓ midas_hmeassetactivitytype → midas_hmeassetactivitytypes (10 cols, 39 total in Dataverse)
✓ midas_hmeassetmapping → midas_hmeassetmappings (27 cols, 56 total in Dataverse)
✓ midas_hmetummapping → midas_hmetummappings (17 cols, 46 total in Dataverse)
✓ midas_manualbasemetrics → midas_manu

sync_table_name,primary_key_column
str,str
"""midas_annualplans""","""midas_annualplanid"""
"""midas_basemetrics""","""midas_basemetricid"""
"""midas_dataflows""","""midas_dataflowid"""
"""midas_dataflowdependencies""","""midas_dataflowdependencyid"""
"""midas_eedrillingprojectlocationses""","""midas_eedrillingprojectlocationsid"""
…,…
"""midas_publicholidaies""","""midas_publicholidayid"""
"""midas_recoveries""","""midas_recoveryid"""
"""midas_sitemappings""","""midas_sitemappingid"""


In [15]:
# ─── Print each Dataverse-sourced config entry ────────────────────────────────
for row in config_dv_df.iter_rows(named=True):
    print(row["sync_table_name"])
    print()
    print(row["last_sync_timestamp"] or "(set at runtime)")
    print()
    print(row["databricks_target_table"])
    print()
    print(row["column_list"])
    print()
    print(row["primary_key_column"])
    print()
    print(row["phantom_row_sql"])
    print()
    print("─" * 120)
    print()

# Export
config_dv_df.write_csv("sync_config_table_dataverse.csv")
print("✓ Exported to sync_config_table_dataverse.csv")

midas_annualplans

(set at runtime)

_ogr_azr_syd_dev.mde.midas_annualplan

createdon, midas_annualplan, midas_annualplanid, midas_effectivedatefrom, midas_environment, midas_planmonth, midas_plannamenew, midas_plannedvalue, midas_plansource, midas_sitecode, modifiedon, statecode

midas_annualplanid

SELECT CAST(NULL AS TIMESTAMP) AS createdon, CAST(NULL AS STRING) AS midas_annualplan, CAST(NULL AS STRING) AS midas_annualplanid, CAST(NULL AS TIMESTAMP) AS midas_effectivedatefrom, CAST(NULL AS INT) AS midas_environment, CAST(NULL AS TIMESTAMP) AS midas_planmonth, CAST(NULL AS STRING) AS midas_plannamenew, CAST(NULL AS DECIMAL(38,18)) AS midas_plannedvalue, CAST(NULL AS STRING) AS midas_plansource, CAST(NULL AS STRING) AS midas_sitecode, CAST(NULL AS TIMESTAMP) AS modifiedon, CAST(NULL AS INT) AS statecode WHERE 1=0

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

midas_basemetrics

(set at runtime)

_ogr_azr_syd_d

In [ ]:
# ─── Generate Power Automate concat() expressions for each table ───────────────
import re

def parse_phantom_columns(phantom_sql: str):
    """Extract (col_name, dbx_type) pairs from phantom_row_sql."""
    # Pattern: CAST(NULL AS <TYPE>) AS <name>
    pattern = r"CAST\(NULL AS ([^)]+)\) AS ([a-z_0-9]+)"
    return re.findall(pattern, phantom_sql)


def is_guid_column(col_name: str) -> bool:
    """Heuristic: GUIDs don't need quote escaping."""
    if col_name.startswith("_") and col_name.endswith("_value"):
        return True
    if col_name.endswith("id"):
        return True
    return False


def generate_concat_expr(columns: list[dict]) -> str:
    """Generate Power Automate concat() expression.
    Each column dict has: api_name (Dataverse response key), dbx_name (SQL alias), dbx_type.
    """
    parts = []

    for idx, col in enumerate(columns):
        api_name = col["api_name"]
        dbx_name = col["dbx_name"]
        dbx_type = col["dbx_type"]
        is_last = idx == len(columns) - 1
        comma = "" if is_last else ","
        base_type = dbx_type.split("(")[0].strip().upper()

        if idx == 0:
            prefix = "  'SELECT "
        else:
            prefix = "  ' "

        if base_type == "TIMESTAMP" or base_type == "DATE":
            if dbx_name in ("createdon", "modifiedon"):
                parts.append(f"{prefix}''', item()?['{api_name}'], '''{comma}'")
            else:
                parts.append(
                    f"{prefix}', if(equals(item()?['{api_name}'],null),'NULL',concat('''',item()?['{api_name}'],'''')), '{comma}'"
                )

        elif base_type == "STRING":
            if is_guid_column(dbx_name) or is_guid_column(api_name):
                parts.append(f"{prefix}''', item()?['{api_name}'], '''{comma}'")
            else:
                parts.append(
                    f"{prefix}''', replace(coalesce(item()?['{api_name}'],''),'''',''''''), '''{comma}'"
                )

        elif base_type in ("INT", "BIGINT", "SMALLINT", "TINYINT"):
            if dbx_name == "statecode" and is_last:
                parts.append(
                    f"{prefix}', if(equals(item()?['{api_name}'],null),'0',string(item()?['{api_name}']))"
                )
            else:
                parts.append(
                    f"{prefix}', if(equals(item()?['{api_name}'],null),'NULL',string(item()?['{api_name}'])), '{comma}'"
                )

        elif base_type in ("DECIMAL", "DOUBLE", "FLOAT"):
            parts.append(
                f"{prefix}', if(equals(item()?['{api_name}'],null),'NULL',string(item()?['{api_name}'])), '{comma}'"
            )

        elif base_type == "BOOLEAN":
            parts.append(
                f"{prefix}', if(equals(item()?['{api_name}'],null),'NULL',if(item()?['{api_name}'],'true','false')), '{comma}'"
            )

        else:
            parts.append(
                f"{prefix}''', replace(coalesce(item()?['{api_name}'],''),'''',''''''), '''{comma}'"
            )

    return "concat(\n" + ",\n".join(parts) + "\n)"


# ─── Generate for all tables ──────────────────────────────────────────────────
print("=" * 80)
print("POWER AUTOMATE concat() EXPRESSIONS")
print("=" * 80)

for row in config_dv_df.iter_rows(named=True):
    table_name = row["sync_table_name"]
    # Rebuild column info from the stored mapping
    mapping = eval(row["column_mapping"])  # {api_name: dbx_name}
    phantom = row["phantom_row_sql"]

    # Parse dbx types from phantom_row_sql (uses dbx_names)
    import re
    type_pattern = r"CAST\(NULL AS ([^)]+)\) AS ([a-z_0-9]+)"
    type_map = {name: dtype for dtype, name in re.findall(type_pattern, phantom)}

    columns = []
    for api_name, dbx_name in mapping.items():
        dbx_type = type_map.get(dbx_name, "STRING")
        columns.append({"api_name": api_name, "dbx_name": dbx_name, "dbx_type": dbx_type})

    if not columns:
        continue

    expr = generate_concat_expr(columns)
    print(f"\n{'─' * 80}")
    print(f"TABLE: {table_name}")
    print(f"{'─' * 80}")
    print(expr)
    print()

POWER AUTOMATE concat() EXPRESSIONS

────────────────────────────────────────────────────────────────────────────────
TABLE: midas_annualplans
────────────────────────────────────────────────────────────────────────────────
concat(
  'SELECT ''', item()?['createdon'], ''',',
  ' ''', replace(coalesce(item()?['midas_annualplan'],''),'''',''''''), ''',',
  ' ''', item()?['midas_annualplanid'], ''',',
  ' ', if(equals(item()?['midas_effectivedatefrom'],null),'NULL',concat('''',item()?['midas_effectivedatefrom'],'''')), ',',
  ' ', if(equals(item()?['midas_environment'],null),'NULL',string(item()?['midas_environment'])), ',',
  ' ', if(equals(item()?['midas_planmonth'],null),'NULL',concat('''',item()?['midas_planmonth'],'''')), ',',
  ' ''', replace(coalesce(item()?['midas_plannamenew'],''),'''',''''''), ''',',
  ' ''', replace(coalesce(item()?['midas_plansource'],''),'''',''''''), ''',',
  ' ''', replace(coalesce(item()?['midas_sitecode'],''),'''',''''''), ''',',
  ' ''', item()?['modifie

In [ ]:
# ─── Recreate Databricks tables using phantom_row_sql column names ──────────────
# This fixes the mismatch between Synapse column names (midas_dataflow)
# and Dataverse Web API names (_midas_dataflow_value) used by Power Automate.

for env in environments:
    env_name = env["name"]
    print(f"\n{'═' * 60}")
    print(f"  Recreating tables in: {env_name.upper()} ({env['catalog']})")
    print(f"{'═' * 60}")

    dbx_conn = sql.connect(
        server_hostname=env["server_hostname"],
        http_path=env["http_path"],
        access_token=env["access_token"],
    )
    dbx_cursor = dbx_conn.cursor()

    for row in config_dv_df.iter_rows(named=True):
        table = row["databricks_target_table"].replace(DBX_CATALOG, env["catalog"])
        phantom = row["phantom_row_sql"]

        # CREATE OR REPLACE TABLE ... AS SELECT (empty, just schema)
        create_sql = f"CREATE OR REPLACE TABLE {table} AS {phantom}"
        try:
            dbx_cursor.execute(create_sql)
            print(f"  ✓ {table}")
        except Exception as e:
            print(f"  ✗ {table}: {e}")

    dbx_cursor.close()
    dbx_conn.close()

print("\n✓ All tables recreated with Dataverse-compatible column names")

In [ ]:
# ─── Clean Up ──────────────────────────────────────────────────────────────────
databricks_cursor.close()
databricks_conn.close()
synapse_conn.close()
print("Connections closed.")